In [1]:

# Imports

import helpers.helper_functions as hf
import mne
import os.path as op
from mne.channels import combine_channels
import pandas as pd
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import hilbert
import helpers.test_circ_plot as circ_plot
import gc
import helpers.stc_helper as stc_helper
import time
from pycircstat2.hypothesis import rayleigh_test
import pycircstat2 as cs2
ss = hf.settings_dict()

In [2]:
baseline_tmin = -0.3
baseline_tmax = 0.0
stimulus_tmin = 0.7
stimulus_tmax = 3.7

def mean_resultant_length(angles):
    """r = |mean(exp(i*angles))| — always in [0,1]"""
    return np.abs(np.mean(np.exp(1j * angles)))

def compute_delta_r(baseline_angles, stimulus_angles):
    r_baseline = mean_resultant_length(baseline_angles)
    r_stimulus = mean_resultant_length(stimulus_angles)
    return r_stimulus - r_baseline   # positive = more locking during stimulus


In [3]:
from pycircstat2.hypothesis import wheeler_watson_test, watson_u2_test

for subject_index in ss['subject_idx_list']:

    # loop over each event type
    for event_id in ss['event_id_list']:

        event_name = str(event_id)
        duty_cycle = ss['event_name_list'][event_id - 1]
        subjects_dir = ss['fs_subjects_dir']
        subject = ss['subject_id_list'][subject_index]
        print("loading dataset for subject: ", subject)

        save_dir = Path(ss['hilbert_ref_dir']) / subject / event_name
        save_dir.mkdir(parents=True, exist_ok=True)

        # load hilbert stc data
        hilbert_stc_file = Path(ss['hilbert_ref_dir']) / subject / event_name / f"{subject}-event-{event_name}-hilbert-ret-ref-vol.stc"
        stc = mne.read_source_estimate(hilbert_stc_file)

        stc_baseline = stc.copy().crop(baseline_tmin, baseline_tmax)
        stc_stimulus = stc.copy().crop(stimulus_tmin, stimulus_tmax)


        # stc.data shape: (n_voxels, n_times) — phase differences in radians [-pi, pi]
        n_voxels, n_times = stc.data.shape

        delta_r_map = np.zeros(n_voxels)

        for voxel_idx in range(n_voxels):
            delta_r_map[voxel_idx] = compute_delta_r(
                stc_baseline.data[voxel_idx],
                stc_stimulus.data[voxel_idx]
            )

        print(f"delta_r min  : {delta_r_map.min():.4f}")
        print(f"delta_r max  : {delta_r_map.max():.4f}")
        print(f"delta_r mean : {delta_r_map.mean():.4f}")
        print(f"Voxels > 0   : {(delta_r_map > 0).sum()} / {n_voxels}")

        z_stc       = stc.copy().crop(0, 0.0 + 1/ss['fs'])
        z_stc.data  = delta_r_map[:, np.newaxis]   # (n_voxels, 1)

        z_stc.save(save_dir / f"{subject}-event-{event_name}-z-vol.stc" , overwrite=True)


loading dataset for subject:  0005_3SJ
delta_r min  : -0.7965
delta_r max  : 0.8547
delta_r mean : -0.1038
Voxels > 0   : 484 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.8569
delta_r max  : 0.8674
delta_r mean : -0.0271
Voxels > 0   : 661 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.9286
delta_r max  : 0.9012
delta_r mean : -0.1589
Voxels > 0   : 372 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.7178
delta_r max  : 0.8736
delta_r mean : 0.0670
Voxels > 0   : 787 / 1440
Overwriting existing file.
Writing STC to disk...
Overwriting existing file.
[done]
loading dataset for subject:  0005_3SJ
delta_r min  : -0.8209
delta_r max  : 0.7140
delta_r mean : -0.0846
Voxels > 0   : 543 / 1440
